In [0]:
%run "/Workspace/Repos/sjdgustavo@gmail.com/databricks-etl-pos/ETL_arquitetura_medalhao/00.config/config"

In [0]:
from pyspark.sql import functions as F, Window

df = spark.table(f"{CATALOG}.{SILVER}.economia")

w = Window.orderBy("data")

gold = (
    df
    .withColumn("ipca_ant", F.lag("ipca").over(w))
    .withColumn("boi_ant", F.lag("boi_gordo").over(w))
    .withColumn("variacao_ipca", (F.col("ipca")-F.col("ipca_ant"))/F.col("ipca_ant")*100)
    .withColumn("variacao_boi", (F.col("boi_gordo")-F.col("boi_ant"))/F.col("boi_ant")*100)
    .drop("ipca_ant", "boi_ant")

)

gold.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{GOLD}.insights")

In [0]:
gold.display()